# SGLang 对接外部 API — 实现推理服务调用

**定位：** 展示如何使用多种客户端方式调用 SGLang 推理服务，包括 requests、OpenAI SDK、curl 和 httpx 异步调用，以及错误处理与重试机制。

> **一条命令启动服务：**

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台*

---

## 硬件与环境要求

| 项目 | 最低要求 | 推荐配置 |
|------|---------|----------|
| GPU | NVIDIA（支持 CUDA） | RTX 3060 及以上 |
| 显存 (VRAM) | ≥ 6 GB | ≥ 8 GB |
| 驱动版本 | ≥ 525.0 | 最新稳定版 |
| CUDA 版本 | ≥ 12.1 | 12.4+ |
| Python | 3.9 – 3.12 | 3.10 |
| 操作系统 | Linux / WSL2 | Ubuntu 22.04 |

---

## 依赖安装

推荐使用虚拟环境隔离依赖：

```bash
# 第一步：创建并激活虚拟环境
python -m venv sglang-env
source sglang-env/bin/activate

# 第二步：安装 SGLang 及所需依赖
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0" "httpx>=0.24.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和 CUDA 可用
2. 如需在 Notebook 中安装依赖，运行「可选安装」代码单元格
3. 运行「启动 SGLang 服务」单元格，等待服务就绪
4. 依次运行各种 API 调用方式的单元格
5. 完成后运行「停止服务释放显存」单元格清理资源

---

In [ ]:
import sys  # 导入系统模块
print(f"Python 版本: {sys.version}")  # 打印 Python 版本信息

import torch  # 导入 PyTorch 深度学习框架
print(f"PyTorch 版本: {torch.__version__}")  # 打印 PyTorch 版本
print(f"CUDA 是否可用: {torch.cuda.is_available()}")  # 检查 CUDA 是否可用

if torch.cuda.is_available():  # 如果 CUDA 可用
    print(f"CUDA 版本: {torch.version.cuda}")  # 打印 CUDA 版本号
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")  # 打印第一张 GPU 名称
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3  # 计算总显存（GB）
    print(f"GPU 显存: {vram_gb:.1f} GB")  # 打印显存大小
else:  # CUDA 不可用
    print("⚠️ 未检测到 GPU，请检查驱动和 CUDA 安装")  # 提示用户检查环境

In [ ]:
import subprocess  # 导入子进程模块
import sys  # 导入系统模块

subprocess.check_call([  # 执行 pip 安装命令
    sys.executable, "-m", "pip", "install", "-q",  # 使用当前 Python 解释器静默安装
    "sglang[all]", "openai>=1.0.0", "requests>=2.28.0", "httpx>=0.24.0"  # 安装 SGLang 及依赖（含 httpx）
])  # 安装命令执行完成
print("✅ 依赖安装完成")  # 打印安装成功提示

## SGLang 作为 OpenAI 兼容 API 服务

SGLang 启动后会提供与 OpenAI API 完全兼容的 HTTP 接口，支持以下端点：

| 端点 | 功能 | HTTP 方法 |
|------|------|----------|
| `/v1/chat/completions` | 聊天补全 | POST |
| `/v1/completions` | 文本补全 | POST |
| `/v1/models` | 查看模型列表 | GET |

这意味着任何支持 OpenAI API 的客户端（SDK、HTTP 库、命令行工具）都可以直接对接 SGLang。

> **提示**：如需从其他机器访问，启动服务时使用 `--host 0.0.0.0` 监听所有网络接口。本教程为安全起见使用 `127.0.0.1`。

## 启动 SGLang 服务

在后台启动 SGLang 推理服务，并等待服务就绪。

In [ ]:
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求库
import sys  # 导入系统模块

server_process = subprocess.Popen(  # 以子进程方式启动 SGLang 推理服务
    [  # 命令参数列表
        sys.executable, "-m", "sglang.launch_server",  # 调用 SGLang 服务启动模块
        "--model-path", "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型路径（首次自动下载）
        "--host", "127.0.0.1",  # 监听本机回环地址
        "--port", "30000",  # 指定服务端口号
    ],  # 命令参数结束
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE,  # 捕获标准错误输出
)  # 子进程创建完成
print(f"SGLang 服务进程已启动，PID = {server_process.pid}")  # 打印进程 ID

max_wait = 600  # 最大等待时间 600 秒（含模型下载时间）
start_time = time.time()  # 记录开始时间
ready = False  # 服务就绪标志初始化为 False

while time.time() - start_time < max_wait:  # 在超时范围内持续轮询
    try:  # 尝试连接服务
        r = requests.get("http://127.0.0.1:30000/v1/models", timeout=5)  # 向模型列表接口发请求
        if r.status_code == 200:  # 返回 200 表示服务已就绪
            print(f"✅ SGLang 服务就绪！耗时 {time.time() - start_time:.1f} 秒")  # 打印就绪信息
            ready = True  # 更新就绪标志
            break  # 退出轮询循环
    except requests.ConnectionError:  # 连接失败说明服务尚未启动完成
        pass  # 忽略异常，继续等待
    time.sleep(2)  # 每隔 2 秒重试一次

if not ready:  # 如果超时仍未就绪
    print("❌ 服务启动超时，请检查 GPU 显存或查看进程日志")  # 打印超时错误信息

## 方式一：使用 requests 库直接调用

使用 Python `requests` 库直接向 SGLang 的 HTTP 端点发送 POST 请求，完全手动构造请求体。

In [ ]:
import requests  # 导入 HTTP 请求库
import json  # 导入 JSON 处理模块

api_url = "http://127.0.0.1:30000/v1/chat/completions"  # 定义聊天补全 API 地址

headers = {  # 定义请求头
    "Content-Type": "application/json",  # 指定请求体为 JSON 格式
    "Authorization": "Bearer EMPTY",  # 本地服务使用占位符认证
}  # 请求头定义完成

request_body = {  # 构造请求体
    "model": "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
    "messages": [  # 消息列表
        {"role": "system", "content": "你是一个有帮助的AI助手。"},  # 系统角色提示
        {"role": "user", "content": "用一句话解释什么是 API"},  # 用户提问
    ],  # 消息列表结束
    "max_tokens": 256,  # 最大生成 token 数
    "temperature": 0.7,  # 温度参数，控制随机性
}  # 请求体构造完成

print("📤 发送请求...")  # 打印发送提示
print(f"   URL: {api_url}")  # 打印请求地址
print(f"   请求体: {json.dumps(request_body, ensure_ascii=False, indent=2)}")  # 打印格式化的请求体

response = requests.post(api_url, headers=headers, json=request_body, timeout=30)  # 发送 POST 请求
print(f"\n📥 响应状态码: {response.status_code}")  # 打印响应状态码

if response.status_code == 200:  # 如果请求成功
    result = response.json()  # 解析 JSON 响应
    content = result["choices"][0]["message"]["content"]  # 提取回复内容
    usage = result.get("usage", {})  # 提取 token 使用信息
    print(f"🤖 回复: {content}")  # 打印模型回复
    print(f"📊 Token 使用: 输入={usage.get('prompt_tokens', 'N/A')}, 输出={usage.get('completion_tokens', 'N/A')}")  # 打印 token 统计
else:  # 请求失败
    print(f"❌ 请求失败: {response.text}")  # 打印错误详情

## 方式二：使用 OpenAI SDK 调用

使用官方 `openai` Python SDK，只需修改 `base_url` 即可无缝对接 SGLang 服务。这是最推荐的调用方式。

In [ ]:
from openai import OpenAI  # 导入 OpenAI 客户端库

client = OpenAI(  # 创建 OpenAI 兼容客户端实例
    base_url="http://127.0.0.1:30000/v1",  # 指向本地 SGLang 服务地址
    api_key="EMPTY",  # 本地服务无需真实 API Key
)  # 客户端创建完成

print("📋 可用模型列表：")  # 打印模型列表标题
models = client.models.list()  # 获取可用模型列表
for model in models.data:  # 遍历所有模型
    print(f"  - {model.id}")  # 打印模型 ID

print("\n📤 使用 OpenAI SDK 发送请求...")  # 打印发送提示
response = client.chat.completions.create(  # 发送聊天补全请求
    model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
    messages=[  # 构造对话消息列表
        {"role": "system", "content": "你是一个有帮助的AI助手。"},  # 系统角色提示
        {"role": "user", "content": "Python 有哪些常用的 Web 框架？"},  # 用户提问
    ],  # 消息列表结束
    max_tokens=256,  # 限制最大生成 token 数
    temperature=0.7,  # 温度参数控制生成随机性
)  # 请求完成

print(f"🤖 回复: {response.choices[0].message.content}")  # 打印模型回复
print(f"📊 Token 使用: 输入={response.usage.prompt_tokens}, 输出={response.usage.completion_tokens}")  # 打印 token 统计

## 方式三：使用 curl 命令调用

生成 curl 命令供终端使用，同时通过 `subprocess` 在 Notebook 内执行并展示结果。

In [ ]:
import subprocess  # 导入子进程模块
import json  # 导入 JSON 处理模块

curl_command = """curl -s http://127.0.0.1:30000/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer EMPTY" \\
  -d '{
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [
      {"role": "system", "content": "你是一个有帮助的AI助手。"},
      {"role": "user", "content": "什么是 RESTful API？"}
    ],
    "max_tokens": 256
  }'"""  # 定义完整的 curl 命令字符串

print("📋 可复制的 curl 命令：")  # 打印标题
print(curl_command)  # 打印 curl 命令供用户复制
print()  # 打印空行

actual_command = [  # 定义实际执行的 curl 参数列表
    "curl", "-s",  # curl 命令，静默模式
    "http://127.0.0.1:30000/v1/chat/completions",  # 请求地址
    "-H", "Content-Type: application/json",  # 设置内容类型头
    "-H", "Authorization: Bearer EMPTY",  # 设置认证头
    "-d", json.dumps({  # 请求体（JSON 字符串）
        "model": "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型
        "messages": [  # 消息列表
            {"role": "system", "content": "你是一个有帮助的AI助手。"},  # 系统提示
            {"role": "user", "content": "什么是 RESTful API？"},  # 用户提问
        ],  # 消息列表结束
        "max_tokens": 256,  # 最大生成 token 数
    }, ensure_ascii=False),  # 转为 JSON 字符串，保留中文
]  # 参数列表定义完成

print("🔄 在 Notebook 中执行 curl 命令...")  # 打印执行提示
result = subprocess.run(actual_command, capture_output=True, text=True, timeout=30)  # 执行 curl 命令

if result.returncode == 0:  # 如果命令执行成功
    response_json = json.loads(result.stdout)  # 解析 JSON 响应
    content = response_json["choices"][0]["message"]["content"]  # 提取回复内容
    print(f"🤖 回复: {content}")  # 打印模型回复
else:  # 命令执行失败
    print(f"❌ curl 执行失败: {result.stderr}")  # 打印错误信息

## 方式四：异步批量调用（httpx）

使用 `httpx.AsyncClient` 进行异步 HTTP 请求，适合需要高并发调用 API 的场景。在 Jupyter Notebook 中使用 `nest_asyncio` 解决事件循环冲突。

In [ ]:
import httpx  # 导入异步 HTTP 客户端库
import asyncio  # 导入异步编程模块
import json  # 导入 JSON 处理模块
import time  # 导入时间模块

try:  # 尝试导入 nest_asyncio
    import nest_asyncio  # 导入嵌套事件循环补丁
    nest_asyncio.apply()  # 应用补丁以允许 Jupyter 中运行 asyncio
except ImportError:  # 如果未安装 nest_asyncio
    print("⚠️ 未安装 nest_asyncio，尝试 pip install nest_asyncio")  # 打印安装提示

api_url = "http://127.0.0.1:30000/v1/chat/completions"  # 定义 API 地址


async def async_chat_request(client, prompt, request_id):  # 定义异步请求函数
    """发送单条异步聊天请求"""
    payload = {  # 构造请求体
        "model": "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
        "messages": [  # 消息列表
            {"role": "system", "content": "你是一个有帮助的AI助手，请简洁回答。"},  # 系统提示
            {"role": "user", "content": prompt},  # 用户提问
        ],  # 消息列表结束
        "max_tokens": 128,  # 最大生成 token 数
    }  # 请求体构造完成
    start = time.time()  # 记录请求开始时间
    response = await client.post(api_url, json=payload, timeout=30.0)  # 异步发送 POST 请求
    elapsed = time.time() - start  # 计算请求耗时
    result = response.json()  # 解析 JSON 响应
    content = result["choices"][0]["message"]["content"]  # 提取回复内容
    return {"id": request_id, "content": content, "latency": elapsed}  # 返回结果字典


async def batch_async_requests(prompts):  # 定义批量异步请求函数
    """批量发送异步请求并收集结果"""
    async with httpx.AsyncClient() as client:  # 创建异步 HTTP 客户端
        tasks = [  # 创建所有异步任务
            async_chat_request(client, prompt, i)  # 为每条提示创建异步任务
            for i, prompt in enumerate(prompts)  # 遍历所有提示语
        ]  # 任务列表创建完成
        results = await asyncio.gather(*tasks)  # 并发执行所有任务并等待完成
    return results  # 返回所有结果


async_prompts = [  # 定义异步测试提示语
    "什么是异步编程？",  # 提示 1
    "HTTP 和 HTTPS 有什么区别？",  # 提示 2
    "什么是 JSON 格式？",  # 提示 3
    "解释一下 DNS 的作用",  # 提示 4
    "什么是缓存？",  # 提示 5
]  # 提示语列表定义完成

print(f"🚀 使用 httpx 异步发送 {len(async_prompts)} 条请求...")  # 打印开始提示
start_time = time.time()  # 记录整体开始时间

results = asyncio.run(batch_async_requests(async_prompts))  # 运行异步批量请求

total_time = time.time() - start_time  # 计算总耗时
print(f"\n✅ 全部完成！总耗时: {total_time:.2f} 秒\n")  # 打印总耗时

for r in results:  # 遍历所有结果
    print(f"  📨 请求 {r['id']}: (耗时 {r['latency']:.3f}s)")  # 打印请求序号和耗时
    print(f"     🤖 {r['content'][:80]}...\n")  # 打印回复前 80 字符

## 错误处理与重试机制

在生产环境中，网络波动和服务端临时故障是常见的。下面封装一个带有重试逻辑的请求函数，支持指数退避和多种错误处理。

In [ ]:
import requests  # 导入 HTTP 请求库
import time  # 导入时间模块
import json  # 导入 JSON 处理模块


def robust_chat_request(  # 定义带重试机制的请求函数
    prompt,  # 用户提示语
    base_url="http://127.0.0.1:30000",  # 服务基础地址
    model="Qwen/Qwen2.5-0.5B-Instruct",  # 模型名称
    max_retries=3,  # 最大重试次数
    base_delay=1.0,  # 初始退避延迟（秒）
    timeout=30,  # 请求超时时间（秒）
):  # 参数定义结束
    """带重试和指数退避的聊天请求函数"""
    url = f"{base_url}/v1/chat/completions"  # 构造完整 API 地址
    headers = {"Content-Type": "application/json"}  # 设置请求头
    payload = {  # 构造请求体
        "model": model,  # 模型名称
        "messages": [  # 消息列表
            {"role": "user", "content": prompt},  # 用户消息
        ],  # 消息列表结束
        "max_tokens": 256,  # 最大生成 token 数
    }  # 请求体构造完成

    for attempt in range(max_retries + 1):  # 循环尝试（含首次 + 重试次数）
        try:  # 尝试发送请求
            response = requests.post(  # 发送 POST 请求
                url, headers=headers, json=payload, timeout=timeout  # 传入地址、头、请求体和超时
            )  # 请求发送完成

            if response.status_code == 200:  # 请求成功
                result = response.json()  # 解析 JSON 响应
                return {  # 返回成功结果
                    "success": True,  # 成功标志
                    "content": result["choices"][0]["message"]["content"],  # 回复内容
                    "attempts": attempt + 1,  # 实际尝试次数
                }  # 结果字典结束

            elif response.status_code >= 500:  # 服务端错误（5xx）
                print(f"  ⚠️ 第 {attempt + 1} 次尝试: 服务端错误 {response.status_code}")  # 打印服务端错误
            elif response.status_code == 422:  # 请求格式错误
                return {  # 格式错误不重试，直接返回
                    "success": False,  # 失败标志
                    "error": f"请求格式错误(422): {response.text}",  # 错误信息
                    "attempts": attempt + 1,  # 实际尝试次数
                }  # 结果字典结束
            else:  # 其他 HTTP 错误
                print(f"  ⚠️ 第 {attempt + 1} 次尝试: HTTP {response.status_code}")  # 打印错误状态码

        except requests.exceptions.Timeout:  # 请求超时异常
            print(f"  ⚠️ 第 {attempt + 1} 次尝试: 请求超时")  # 打印超时信息
        except requests.exceptions.ConnectionError:  # 连接错误异常
            print(f"  ⚠️ 第 {attempt + 1} 次尝试: 连接失败")  # 打印连接失败信息
        except Exception as e:  # 其他未预期的异常
            print(f"  ⚠️ 第 {attempt + 1} 次尝试: 未知错误 - {e}")  # 打印未知错误信息

        if attempt < max_retries:  # 如果还有剩余重试次数
            delay = base_delay * (2 ** attempt)  # 计算指数退避延迟
            print(f"  ⏳ 等待 {delay:.1f} 秒后重试...")  # 打印等待信息
            time.sleep(delay)  # 等待指定时间

    return {  # 所有重试均失败，返回最终失败结果
        "success": False,  # 失败标志
        "error": f"在 {max_retries + 1} 次尝试后仍然失败",  # 错误信息
        "attempts": max_retries + 1,  # 总尝试次数
    }  # 最终结果字典结束


# ---- 测试正常请求 ----
print("✅ 测试正常请求：")  # 打印正常请求测试标题
result = robust_chat_request("你好，请做个自我介绍")  # 发送正常请求
if result["success"]:  # 如果请求成功
    print(f"  🤖 回复: {result['content']}")  # 打印回复内容
    print(f"  📊 尝试次数: {result['attempts']}")  # 打印尝试次数

# ---- 测试错误地址（触发重试） ----
print("\n❌ 测试错误地址（将触发重试）：")  # 打印错误地址测试标题
result_fail = robust_chat_request(  # 向错误地址发送请求
    "测试",  # 测试提示语
    base_url="http://127.0.0.1:39999",  # 使用不存在的端口
    max_retries=2,  # 最多重试 2 次
    base_delay=0.5,  # 退避延迟 0.5 秒（加快演示速度）
)  # 请求发送完成
print(f"  结果: {result_fail}")  # 打印最终结果

## 停止服务释放显存

In [ ]:
import os  # 导入操作系统模块
import signal  # 导入信号处理模块

if 'server_process' in dir() and server_process.poll() is None:  # 检查服务进程是否仍在运行
    os.kill(server_process.pid, signal.SIGTERM)  # 向服务进程发送终止信号
    server_process.wait(timeout=10)  # 等待进程退出，最多等 10 秒
    print(f"✅ SGLang 服务已停止 (PID={server_process.pid})")  # 打印停止成功信息
else:  # 服务进程不存在或已结束
    print("ℹ️ 服务进程未运行或已结束")  # 打印提示信息

if 'torch' in dir() and torch.cuda.is_available():  # 检查 PyTorch 和 CUDA 是否可用
    torch.cuda.empty_cache()  # 清空 GPU 显存缓存
    print("✅ GPU 显存缓存已清理")  # 打印显存清理完成信息

## 常见问题排查

### 1. Connection Refused（连接被拒绝）

**现象**：请求时报错 `ConnectionError: Connection refused`

**原因**：SGLang 服务未启动，或端口号不匹配。

**解决**：
- 确认 SGLang 服务已启动并且显示「服务就绪」提示
- 检查请求中使用的端口号与启动命令中的 `--port` 一致
- 如果服务运行在远程机器上，确认网络可达且防火墙已放行该端口

### 2. 422 Unprocessable Entity（请求体格式错误）

**现象**：请求返回 HTTP 422 错误。

**原因**：请求体 JSON 格式不符合 OpenAI API 规范。

**解决**：
- 检查请求体必须包含 `model` 和 `messages` 字段
- `messages` 必须是包含 `role` 和 `content` 的字典列表
- `role` 只能是 `system`、`user`、`assistant` 之一
- 参考 OpenAI 官方 API 文档确认请求格式

---

**结语**：本教程展示了四种调用 SGLang 推理服务的方式：requests 原生调用、OpenAI SDK、curl 命令行、httpx 异步调用。在生产环境中，推荐使用 OpenAI SDK（代码简洁）或 httpx 异步方式（高并发性能好），并搭配重试机制提升服务可靠性。